In [1]:
"""
A full tutorial is available here:
https://nichecompass.readthedocs.io/en/latest/tutorials/notebooks/mouse_cns_spatial_reference_mapping.html

Here we illustrate preaparation of our core dataset as a reference for mapping the validation data
"""

'\nA full tutorial is available here:\nhttps://nichecompass.readthedocs.io/en/latest/tutorials/notebooks/mouse_cns_spatial_reference_mapping.html\n\nHere we illustrate preaparation of our core dataset as a reference for mapping the validation data\n'

In [2]:
import os, sys
import random
import warnings
import logging
from datetime import datetime
# import gdown
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import seaborn as sns
import squidpy as sq
#from matplotlib import gridspec
#from sklearn.preprocessing import MinMaxScaler
from re import sub
import numpy as np
import pickle

# from nichecompass.models import NicheCompass
# from nichecompass.utils import (add_gps_from_gp_dict_to_adata,
#                                 create_new_color_dict,
#                                 compute_communication_gp_network,
#                                 visualize_communication_gp_network,
#                                 extract_gp_dict_from_mebocost_ms_interactions,
#                                 #extract_gp_dict_from_mebocost_es_interactions,
#                                 extract_gp_dict_from_nichenet_lrt_interactions,
#                                 extract_gp_dict_from_omnipath_lr_interactions,
#                                 #filter_and_combine_gp_dict_gps,
#                                 filter_and_combine_gp_dict_gps_v2,
#                                 generate_enriched_gp_info_plots)


# %%


/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/utils.py:429: 

# Set output dir + where to find niche compass files

In [3]:
"""
make sure inside this path, you have the folders gene_annotations 
and gene_programs with the files
(available from https://github.com/Lotfollahi-lab/nichecompass/tree/main/data)
"""

handle='/lustre/scratch124/cellgen/haniffa/projects/developmental_fibroblasts/nobackup_output/nichecompasss/nichecompass/' 


# Choose number of SVGs to reduce dataset to

In [4]:
n_svg=1024


# Set up reference and query (important part)

In [5]:
"""
load adata (includes reference and query)
- note that sample id is in adata.obs["Sample"]
- cell type is in adata.obs["Annotation"]

if using our adata as reference, then either:
1. remove all query samples (in query_batches below), or
2. add query samples to reference_batches, 

and then add your sample id's to query_batches
"""

#ADATA_PATH= '/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/adata_all.h5ad.clustered.clustered10.good.prenichecompass'
#'/nfs/team298/ls34/xenium_atlas/model_ALL_CLEAN_scanvi_ALL/adata_counts_integrated_final_colored.h5ad'
ADATA_PATH = '/nfs/team298/ls34/adult_skin/final_adatas/adata_combined_new.h5ad.final.filtered'
adata_vis=sc.read_h5ad(ADATA_PATH)  
adata_vis=adata_vis[adata_vis.obs["tech"]=="xenium"].copy()







In [6]:
#adata=sc.read_h5ad('/nfs/team298/ls34/adult_skin/final_adatas/adata_newtime_mintflow.h5ad')
#adata.obs["info_id6"]=adata.obs["sample_id"]
#adata=sc.read_h5ad('/nfs/team361/ls34/inflow/adata_hs_spatial_5k.h5ad')
adata=sc.read_h5ad('/lustre/scratch124/cellgen/haniffa/projects/adult_skin_v1/nobackup_output/adata_relapse_xenium.h5ad')


In [7]:
adata.obs.info_id6.isna().sum()

0

In [8]:
adata.obs["Annotation"]=adata.obs["scanvi_predictions"]
adata.obs["Annotation"].value_counts()

Annotation
F2: Universal       51233
KC3                 37507
T                   32647
KC1                 32262
VE3_Ven             16541
                    ...  
VE3_Ven_APLN+           1
TRM_IL17+               1
ILC1_NCR2+P2RX7+        1
ILC2                    1
Tc2                     1
Name: count, Length: 93, dtype: int64

In [9]:
adata.obs["sample"]=adata.obs["info_id6"]
adata.obs["sample"].value_counts()

sample
BK72_Relapse                         27877
BK74_Baseline never lesional         24599
BK70_Week 8 past lesional            20903
BK74_Relapse                         18584
BK74_Relapse (past lesional site)    17601
BK74_Baseline past lesional          17381
BK70_Baseline past lesional          16999
BK73_Week 4 past lesional            16551
BK72_Baseline past lesional          16497
BK74_Week 4 past lesional            15939
BK70_Relapse                         15111
BK73_Relapse                         14790
BK73_Baseline past lesional          14358
BK70_Baseline never lesional         13737
BK70_Week 4 past lesional            11457
BK72_Relapse (past lesional site)     9403
BK72_Week 4 past lesional             9115
BK73_Baseline never lesional          7602
BK72_Baseline never lesional          6631
Name: count, dtype: int64

In [10]:
adata_vis.obs["sample"]=adata_vis.obs["info_id6"]
adata_vis.obs["sample"].value_counts()

sample
3D_BK25_week12-D2                                  41325
BK39_Week 12                                       32058
BK30_Day 14                                        30438
Healthy1                                           29833
Healthy2                                           29482
                                                   ...  
Baseline_resolved_CE6-SKI-20-FO-1-S22-C2            6046
Baseline_resolved_CE6-SKI-20-FO-1-S22_replicate     5853
Baseline_resolved_CE4-SKI-27-FO-1-S22-B2            5642
Baseline_resolved_CE4-SKI-27-FO-1-S22_replicate     5477
BK20_Lesional Baseline                              2290
Name: count, Length: 97, dtype: int64

In [11]:
adata_vis=ad.concat([adata_vis, adata],
                   label="batch_nc",
                    keys=["reference", "query"],
                  #  join="outer"
                   )


In [12]:
adata_vis.obs["Annotation"].value_counts()

Annotation
KC1                 138572
KC3                 138255
F2: Universal       130826
T                    67420
VE3_Ven              60781
                     ...  
Mac2_CCL14hi             3
Tc2                      1
TransitionalDC           1
KC_HF: IRS               1
ILC1_NCR2+P2RX7+         1
Name: count, Length: 115, dtype: int64

In [13]:
"""
SPLIT INTO REFERENCE AND QUERY
"""
 
"""
replace these query batches with your samples
"""
query_batches = adata.obs["sample"].unique().to_list()


reference_batches = [x for x in adata_vis.obs["sample"].unique() if x not in query_batches]
query_batches

/tmp/ipykernel_892617/2895834108.py:8: FutureWarning: Categorical.to_list is deprecated and will be removed in a future version. Use obj.tolist() instead
  query_batches = adata.obs["sample"].unique().to_list()


['BK73_Week 4 past lesional',
 'BK74_Baseline never lesional',
 'BK72_Baseline past lesional',
 'BK72_Baseline never lesional',
 'BK72_Week 4 past lesional',
 'BK72_Relapse',
 'BK74_Relapse (past lesional site)',
 'BK73_Baseline past lesional',
 'BK72_Relapse (past lesional site)',
 'BK70_Relapse',
 'BK70_Baseline never lesional',
 'BK70_Week 8 past lesional',
 'BK74_Baseline past lesional',
 'BK73_Relapse',
 'BK70_Baseline past lesional',
 'BK73_Baseline never lesional',
 'BK74_Week 4 past lesional',
 'BK74_Relapse',
 'BK70_Week 4 past lesional']

In [14]:
# adata_vis.obs["batch_nc"] = "query"
# adata_vis.obs.loc[adata_vis.obs["sample"].isin(reference_batches), "batch_nc"] = "reference"

In [15]:
adata_vis.layers["counts"]=adata_vis.X.copy()

In [16]:
adata_vis

AnnData object with n_obs × n_vars = 1723590 × 4993
    obs: 'sample_id', 'barcode', 'GSE', 'Site_status', 'Patient_status', 'Location', 'Age', 'Sex', 'dataset_id', 'n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'lvl5_annotation', 'Mapping_status', 'scanvi_predictions', 'atlas_status', 'atlas_status_reynolds', 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_150_genes', 'n_counts', 'Sanger patient ID', 'Timepoint', 'Annotation', 'sample', 'info_id6', 'tech', 'scanvi_predictions2', 'lvl1_new', 'batch_nc'
    obsm: 'X_scvi', 'X_umap', 'spatial'
    layers: 'counts'

In [17]:
query_check = list(adata_vis[adata_vis.obs["batch_nc"]=="query"].obs["sample"].unique())
query_check

['BK73_Week 4 past lesional',
 'BK74_Baseline never lesional',
 'BK72_Baseline past lesional',
 'BK72_Baseline never lesional',
 'BK72_Week 4 past lesional',
 'BK72_Relapse',
 'BK74_Relapse (past lesional site)',
 'BK73_Baseline past lesional',
 'BK72_Relapse (past lesional site)',
 'BK70_Relapse',
 'BK70_Baseline never lesional',
 'BK70_Week 8 past lesional',
 'BK74_Baseline past lesional',
 'BK73_Relapse',
 'BK70_Baseline past lesional',
 'BK73_Baseline never lesional',
 'BK74_Week 4 past lesional',
 'BK74_Relapse',
 'BK70_Week 4 past lesional']

In [18]:
query_check = list(adata_vis[adata_vis.obs["batch_nc"]=="query"].obs["sample"].unique())
for x in query_batches:
    if x not in query_check:
        raise ValueError(f"Batch '{x}' not found in query samples.")

In [19]:
reference_batches= list(adata_vis[adata_vis.obs["batch_nc"]=="reference"].obs["sample"].unique())
query_batches= list(adata_vis[adata_vis.obs["batch_nc"]=="query"].obs["sample"].unique())

In [20]:
all_batches = reference_batches +query_batches


In [21]:
### from jimmy lee 
def select_slide2(adata, s, s_col='sample'):
    """ This function selects the data for one slide from the spatial anndata object.
    :param adata: Anndata object with multiple spatial experiments
    :param s: name of selected experiment
    :param s_col: column in adata.obs listing experiment name for each location
    """
    slide = adata[adata.obs[s_col].isin([s]), :]
#     s_keys = list(slide.uns['spatial'].keys())
#     s_spatial = np.array(s_keys)[[s in k for k in s_keys]][0]
#     slide.uns['spatial'] = {s_spatial: slide.uns['spatial'][s_spatial]}
    return slide

In [22]:
spatial_key = "spatial"
n_neighbors = 8
adj_key = "spatial_connectivities"

adata_batch_list = []
print("Processing reference batches...")
for batch in all_batches:
    print(f"Processing batch {batch}...")
    print("Loading data...")
    adata_batch = select_slide2(adata_vis, batch)
    print(f"Size {adata_batch.shape}")
    print("Computing spatial neighborhood graph...\n")
    # Compute (separate) spatial neighborhood graphs
    logging.info("sq.gr.spatial_neighbors")
    #try:
    sq.gr.spatial_neighbors(adata_batch,
                                coord_type="generic",
                                spatial_key=spatial_key,
                                n_neighs=n_neighbors)
    #except:
    #    continue
    print(f"Spatial neighbours done ## {adata_batch.shape}")

    # Make adjacency matrix symmetric
    adata_batch.obsp[adj_key] = (
        adata_batch.obsp[adj_key].maximum(
            adata_batch.obsp[adj_key].T))
    adata_batch_list.append(adata_batch)
print("List made: ...")
for x in adata_batch_list:
    print(x.shape)
adata_vis = ad.concat(adata_batch_list, join="inner")

Processing reference batches...
Processing batch BK50_Past Lesional...
Loading data...
Size (9800, 4993)
Computing spatial neighborhood graph...

Spatial neighbours done ## (9800, 4993)
Processing batch BK49_Past Lesional wk8 relaspe...
Loading data...
Size (12682, 4993)
Computing spatial neighborhood graph...



/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data


Spatial neighbours done ## (12682, 4993)
Processing batch BK46_Never Lesional...
Loading data...
Size (18754, 4993)
Computing spatial neighborhood graph...

Spatial neighbours done ## (18754, 4993)
Processing batch BK51_Past Lesional wk8 relaspe...
Loading data...
Size (10690, 4993)
Computing spatial neighborhood graph...

Spatial neighbours done ## (10690, 4993)


/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data


Processing batch BK51_wk8 Relapse...
Loading data...
Size (14433, 4993)
Computing spatial neighborhood graph...

Spatial neighbours done ## (14433, 4993)
Processing batch BK46_Past Lesional...
Loading data...
Size (18353, 4993)
Computing spatial neighborhood graph...



...

/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data


Spatial neighbours done ## (17381, 4993)
Processing batch BK73_Relapse...
Loading data...
Size (14790, 4993)
Computing spatial neighborhood graph...

Spatial neighbours done ## (14790, 4993)
Processing batch BK70_Baseline past lesional...
Loading data...
Size (16999, 4993)
Computing spatial neighborhood graph...



/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data


Spatial neighbours done ## (16999, 4993)
Processing batch BK73_Baseline never lesional...
Loading data...
Size (7602, 4993)
Computing spatial neighborhood graph...

Spatial neighbours done ## (7602, 4993)
Processing batch BK74_Week 4 past lesional...
Loading data...
Size (15939, 4993)
Computing spatial neighborhood graph...

Spatial neighbours done ## (15939, 4993)
Processing batch BK74_Relapse...
Loading data...
Size (18584, 4993)
Computing spatial neighborhood graph...

Spatial neighbours done ## (18584, 4993)
Processing batch BK70_Week 4 past lesional...
Loading data...
Size (11457, 4993)
Computing spatial neighborhood graph...

Spatial neighbours done ## (11457, 4993)
List made: ...
(9800, 4993)
(12682, 4993)
(18754, 4993)
(10690, 4993)
(14433, 4993)
(18353, 4993)
(6883, 4993)
(12110, 4993)
(11403, 4993)
(27387, 4993)
(13472, 4993)
(15454, 4993)
(14673, 4993)
(12627, 4993)
(21567, 4993)
(11247, 4993)
(17684, 4993)
(24433, 4993)
(9152, 4993)
(13422, 4993)
(15508, 4993)
(2290, 4993)


/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data


In [23]:
# Combine spatial neighborhood graphs as disconnected components
batch_connectivities = []
len_before_batch = 0
for i in range(len(adata_batch_list)):
    if i == 0: # first batch
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[0].shape[0],
            (adata_vis.shape[0] -
            adata_batch_list[0].shape[0])))
        batch_connectivities.append(sp.hstack(
            (adata_batch_list[0].obsp[adj_key],
            after_batch_connectivities_extension)))
    elif i == (len(adata_batch_list) - 1): # last batch
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_vis.shape[0] -
            adata_batch_list[i].shape[0])))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp[adj_key])))
    else: # middle batches
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0], len_before_batch))
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_vis.shape[0] -
            adata_batch_list[i].shape[0] -
            len_before_batch)))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp[adj_key],
            after_batch_connectivities_extension)))
    len_before_batch += adata_batch_list[i].shape[0]
adata_vis.obsp[adj_key] = sp.vstack(batch_connectivities)



In [24]:
sq.gr.spatial_autocorr(adata_vis, mode="moran", genes=adata_vis.var_names)


In [25]:
sv_genes = adata_vis.uns["moranI"].index[:n_svg].tolist()


In [26]:
adata_vis.var["spatially_variable"] = adata_vis.var_names.isin(sv_genes)
adata_vis.var["keep_gene"] = adata_vis.var["spatially_variable"]
adata_vis = adata_vis[:, adata_vis.var["keep_gene"] == True]
print(f"Keeping {len(adata_vis.var_names)} spatially variable, highly "
       "variable or gene program relevant genes.")


Keeping 1024 spatially variable, highly variable or gene program relevant genes.


In [27]:
import pickle
filepath = f"{handle}/svgenelist.pkl"
with open(filepath, "wb") as f:
    pickle.dump(sv_genes, f)

In [28]:
# sv_genes

In [29]:
adata_vis

View of AnnData object with n_obs × n_vars = 1723590 × 1024
    obs: 'sample_id', 'barcode', 'GSE', 'Site_status', 'Patient_status', 'Location', 'Age', 'Sex', 'dataset_id', 'n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'lvl5_annotation', 'Mapping_status', 'scanvi_predictions', 'atlas_status', 'atlas_status_reynolds', 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_150_genes', 'n_counts', 'Sanger patient ID', 'Timepoint', 'Annotation', 'sample', 'info_id6', 'tech', 'scanvi_predictions2', 'lvl1_new', 'batch_nc'
    var: 'spatially_variable', 'keep_gene'
    uns: 'moranI'
    obsm: 'X_scvi', 'X_umap', 'spatial'
    layers: 'coun

In [30]:
for x in adata_vis.obs.columns:
    adata_vis.obs[x]=    adata_vis.obs[x].astype(str)

/tmp/ipykernel_892617/3822545579.py:2: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_vis.obs[x]=    adata_vis.obs[x].astype(str)


In [31]:
# for x in adata.var.columns:
#     adata.var[x]=    adata.var[x].astype(str)

In [32]:
ADATA_PATH= '/lustre/scratch124/cellgen/haniffa/projects/adult_skin_v1/nobackup_data/adata_core_plus_RELAPSE_refquery2.h5ad'

In [33]:
ADATA_PATH + ".svg"

'/lustre/scratch124/cellgen/haniffa/projects/adult_skin_v1/nobackup_data/adata_core_plus_RELAPSE_refquery2.h5ad.svg'

In [34]:
adata_vis.write(ADATA_PATH + ".svg")  
print(ADATA_PATH + ".svg")

/lustre/scratch124/cellgen/haniffa/projects/adult_skin_v1/nobackup_data/adata_core_plus_RELAPSE_refquery2.h5ad.svg


In [1]:
import scanpy as sc
adata = sc.read_h5ad('/lustre/scratch124/cellgen/haniffa/projects/adult_skin_v1/nobackup_data/adata_core_plus_RELAPSE_refquery2.h5ad.svg')
adata
adata[adata.obs["batch_nc"]=="query"].obs.sample_id.value_counts()

/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/utils.py:429: 

sample_id
output-XETG00272__0080896__BK72-SKI-0-FO-1-s5_BK72-SKI-22-FO-1-s5__20260413__125221     36992
output-XETG00272__0081038__BK73-SKI-26-FO-3-s2_BK74-SKI-27-FO-1-s2__20260413__125221    32171
output-XETG00272__0081038__BK74-SKI-26-FO-1-s2__20260413__125221                        24599
output-XETG00272__0080896__BK70-SKI-22-FO-1-s5_BK72-SKI-26-FO-1-s5__20260413__125221    24514
output-XETG00272__0080896__BK70-SKI-0-FO-1-s5__20260413__125221                         20903
output-XETG00272__0081038__BK74-SKI-27-FO-3-s3__20260413__125221                        18584
output-XETG00272__0081038__BK74-SKI-27-FO-4-s3__20260413__125221                        17601
output-XETG00272__0080896__BK70-SKI-27-FO-1-s5__20260413__125221                        16999
output-XETG00272__0081038__BK73-SKI-26-FO-2-s3__20260413__125221                        16551
output-XETG00272__0080896__BK72-SKI-27-FO-1-s3__20260413__125221                        16497
output-XETG00272__0081038__BK74-SKI-27-FO-2-s3__20

In [60]:
# adata.var

,spatially_variable,keep_gene
A2ML1,True,True
AAMP,True,True
ABCC1,True,True
ABCD3,True,True
ABHD6,True,True
...,...,...
ZC3H12A,True,True
ZCCHC2,True,True
ZMIZ2,True,True
ZNF683,True,True


In [ ]:
9

In [36]:
0

0

# Split into ref + query adatas

In [37]:
# ADATA_PATH= '/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/adata_all.h5ad.clustered.clustered10.good.prenichecompass.svg'
# adata_vis=sc.read_h5ad(ADATA_PATH)  
# adata_vis.shape

In [38]:
query_batches = list(adata_vis[adata_vis.obs["batch_nc"]=="query"].obs["sample"].unique())
reference_batches = list(adata_vis[adata_vis.obs["batch_nc"]=="reference"].obs["sample"].unique())


In [39]:
adata_vis.obs["batch_nc"].value_counts()

batch_nc
reference    1428455
query         295135
Name: count, dtype: int64

In [40]:
adata_batch_list = []
print("Processing reference batches...")
for batch in reference_batches:
    print(f"Processing batch {batch}...")
    adata_batch = select_slide2(adata_vis, batch)
    sq.gr.spatial_neighbors(adata_batch,
                                coord_type="generic",
                                spatial_key=spatial_key,
                                n_neighs=n_neighbors)
    #except:
    #    continue
    print(f"Spatial neighbours done ## {adata_batch.shape}")

    # Make adjacency matrix symmetric
    adata_batch.obsp[adj_key] = (
        adata_batch.obsp[adj_key].maximum(
            adata_batch.obsp[adj_key].T))
    adata_batch_list.append(adata_batch)
print("List made: ...")
for x in adata_batch_list:
    print(x.shape)
adata_reference = ad.concat(adata_batch_list, join="inner")

Processing reference batches...
Processing batch BK50_Past Lesional...
Spatial neighbours done ## (9800, 1024)
Processing batch BK49_Past Lesional wk8 relaspe...


/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data


Spatial neighbours done ## (12682, 1024)
Processing batch BK46_Never Lesional...


/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data


Spatial neighbours done ## (18754, 1024)
Processing batch BK51_Past Lesional wk8 relaspe...
Spatial neighbours done ## (10690, 1024)
Processing batch BK51_wk8 Relapse...


...

Spatial neighbours done ## (20084, 1024)
Processing batch 3D_BK22_Lesional_baseline-B1...


/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data


Spatial neighbours done ## (20444, 1024)
Processing batch 3D_BK22_Lesional_baseline-C2...


/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data


Spatial neighbours done ## (22069, 1024)
Processing batch BK22_Week 12...
Spatial neighbours done ## (9649, 1024)
List made: ...
(9800, 1024)
(12682, 1024)
(18754, 1024)
(10690, 1024)
(14433, 1024)
(18353, 1024)
(6883, 1024)
(12110, 1024)
(11403, 1024)
(27387, 1024)
(13472, 1024)
(15454, 1024)
(14673, 1024)
(12627, 1024)
(21567, 1024)
(11247, 1024)
(17684, 1024)
(24433, 1024)
(9152, 1024)
(13422, 1024)
(15508, 1024)
(2290, 1024)
(7620, 1024)
(15091, 1024)
(8579, 1024)
(7907, 1024)
(17731, 1024)
(6127, 1024)
(14893, 1024)
(8978, 1024)
(6870, 1024)
(16503, 1024)
(16377, 1024)
(32058, 1024)
(13511, 1024)
(30438, 1024)
(11638, 1024)
(13999, 1024)
(16910, 1024)
(13456, 1024)
(18678, 1024)
(11265, 1024)
(12069, 1024)
(12954, 1024)
(9387, 1024)
(11082, 1024)
(6193, 1024)
(5642, 1024)
(8121, 1024)
(11041, 1024)
(18088, 1024)
(22124, 1024)
(11135, 1024)
(12801, 1024)
(8452, 1024)
(9319, 1024)
(11694, 1024)
(9924, 1024)
(15680, 1024)
(14745, 1024)
(20471, 1024)
(6046, 1024)
(8357, 1024)
(10700, 

In [41]:
print("List made: ...", len(adata_batch_list))
for x in adata_batch_list:
    print(x.shape)

List made: ... 97
(9800, 1024)
(12682, 1024)
(18754, 1024)
(10690, 1024)
(14433, 1024)
(18353, 1024)
(6883, 1024)
(12110, 1024)
(11403, 1024)
(27387, 1024)
(13472, 1024)
(15454, 1024)
(14673, 1024)
(12627, 1024)
(21567, 1024)
(11247, 1024)
(17684, 1024)
(24433, 1024)
(9152, 1024)
(13422, 1024)
(15508, 1024)
(2290, 1024)
(7620, 1024)
(15091, 1024)
(8579, 1024)
(7907, 1024)
(17731, 1024)
(6127, 1024)
(14893, 1024)
(8978, 1024)
(6870, 1024)
(16503, 1024)
(16377, 1024)
(32058, 1024)
(13511, 1024)
(30438, 1024)
(11638, 1024)
(13999, 1024)
(16910, 1024)
(13456, 1024)
(18678, 1024)
(11265, 1024)
(12069, 1024)
(12954, 1024)
(9387, 1024)
(11082, 1024)
(6193, 1024)
(5642, 1024)
(8121, 1024)
(11041, 1024)
(18088, 1024)
(22124, 1024)
(11135, 1024)
(12801, 1024)
(8452, 1024)
(9319, 1024)
(11694, 1024)
(9924, 1024)
(15680, 1024)
(14745, 1024)
(20471, 1024)
(6046, 1024)
(8357, 1024)
(10700, 1024)
(6214, 1024)
(5853, 1024)
(7729, 1024)
(20573, 1024)
(17704, 1024)
(9172, 1024)
(10902, 1024)
(5477, 1024

In [42]:
# adata_reference = ad.concat(adata_batch_list, join="inner")


In [43]:
batch_connectivities = []
len_before_batch = 0
for i in range(len(adata_batch_list)):
    if i == 0: # first batch
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[0].shape[0],
            (adata_reference.shape[0] -
            adata_batch_list[0].shape[0])))
        batch_connectivities.append(sp.hstack(
            (adata_batch_list[0].obsp[adj_key],
            after_batch_connectivities_extension)))
    elif i == (len(adata_batch_list) - 1): # last batch
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_reference.shape[0] -
            adata_batch_list[i].shape[0])))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp[adj_key])))
    else: # middle batches
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0], len_before_batch))
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_reference.shape[0] -
            adata_batch_list[i].shape[0] -
            len_before_batch)))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp[adj_key],
            after_batch_connectivities_extension)))
    len_before_batch += adata_batch_list[i].shape[0]
adata_reference.obsp[adj_key] = sp.vstack(batch_connectivities)


In [44]:
mapping_entity_key = "mapping_entity"
adata_reference.obs[mapping_entity_key] = "reference"


In [45]:
adata_reference.write(ADATA_PATH + ".svg.reference")  
print(ADATA_PATH + ".svg.reference")

/lustre/scratch124/cellgen/haniffa/projects/adult_skin_v1/nobackup_data/adata_core_plus_RELAPSE_refquery2.h5ad.svg.reference


In [46]:
adata_vis.obs.batch_nc.value_counts()

batch_nc
reference    1428455
query         295135
Name: count, dtype: int64

In [47]:
query_batches

['BK73_Week 4 past lesional',
 'BK74_Baseline never lesional',
 'BK72_Baseline past lesional',
 'BK72_Baseline never lesional',
 'BK72_Week 4 past lesional',
 'BK72_Relapse',
 'BK74_Relapse (past lesional site)',
 'BK73_Baseline past lesional',
 'BK72_Relapse (past lesional site)',
 'BK70_Relapse',
 'BK70_Baseline never lesional',
 'BK70_Week 8 past lesional',
 'BK74_Baseline past lesional',
 'BK73_Relapse',
 'BK70_Baseline past lesional',
 'BK73_Baseline never lesional',
 'BK74_Week 4 past lesional',
 'BK74_Relapse',
 'BK70_Week 4 past lesional']

In [48]:
adata_batch_list = []
print("Processing query batches...")
# for batch in query_batches:
#     print(f"Processing batch {batch}...")
#     print("Loading data...")
#     adata_batch = sc.read_h5ad(
#         f"{so_data_folder_path}/{dataset}_{batch}.h5ad")
for batch in query_batches:
   # print(f"Processing batch {batch}...")
    #print("Loading data...")
    adata_batch = select_slide2(adata_vis, batch)
    #print("Computing spatial neighborhood graph...\n")
    # Compute (separate) spatial neighborhood graphs
    sq.gr.spatial_neighbors(adata_batch,
                            coord_type="generic",
                            spatial_key=spatial_key,
                            n_neighs=n_neighbors)
    
    # Make adjacency matrix symmetric
    adata_batch.obsp[adj_key] = (
        adata_batch.obsp[adj_key].maximum(
            adata_batch.obsp[adj_key].T))
    adata_batch_list.append(adata_batch)
adata_query = ad.concat(adata_batch_list, join="inner")

Processing query batches...


/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_util

In [49]:
adata_query.obs["sample"].value_counts()

sample
BK72_Relapse                         27877
BK74_Baseline never lesional         24599
BK70_Week 8 past lesional            20903
BK74_Relapse                         18584
BK74_Relapse (past lesional site)    17601
BK74_Baseline past lesional          17381
BK70_Baseline past lesional          16999
BK73_Week 4 past lesional            16551
BK72_Baseline past lesional          16497
BK74_Week 4 past lesional            15939
BK70_Relapse                         15111
BK73_Relapse                         14790
BK73_Baseline past lesional          14358
BK70_Baseline never lesional         13737
BK70_Week 4 past lesional            11457
BK72_Relapse (past lesional site)     9403
BK72_Week 4 past lesional             9115
BK73_Baseline never lesional          7602
BK72_Baseline never lesional          6631
Name: count, dtype: int64

In [50]:
len(adata_batch_list)

19

In [51]:
# adata_batch_list

In [52]:
for i in range(len(adata_batch_list)):
    print(adata_batch_list[i].shape)

(16551, 1024)
(24599, 1024)
(16497, 1024)
(6631, 1024)
(9115, 1024)
(27877, 1024)
(17601, 1024)
(14358, 1024)
(9403, 1024)
(15111, 1024)
(13737, 1024)
(20903, 1024)
(17381, 1024)
(14790, 1024)
(16999, 1024)
(7602, 1024)
(15939, 1024)
(18584, 1024)
(11457, 1024)


In [53]:
# Combine spatial neighborhood graphs as disconnected components
batch_connectivities = []
len_before_batch = 0
for i in range(len(adata_batch_list)):
    if i == 0: # first batch
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[0].shape[0],
            (adata_query.shape[0] -
            adata_batch_list[0].shape[0])))
        batch_connectivities.append(sp.hstack(
            (adata_batch_list[0].obsp[adj_key],
            after_batch_connectivities_extension)))
    elif i == (len(adata_batch_list) - 1): # last batch
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_query.shape[0] -
            adata_batch_list[i].shape[0])))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp[adj_key])))
    else: # middle batches
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0], len_before_batch))
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_query.shape[0] -
            adata_batch_list[i].shape[0] -
            len_before_batch)))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp[adj_key],
            after_batch_connectivities_extension)))
    len_before_batch += adata_batch_list[i].shape[0]
adata_query.obsp[adj_key] = sp.vstack(batch_connectivities)

adata_query.obs[mapping_entity_key] = "query"


In [54]:
batch_connectivities = []
len_before_batch = 0

adata_query.obs[mapping_entity_key] = "query"

In [55]:
adata_query.write(ADATA_PATH + ".svg.query")  
print(ADATA_PATH + ".svg.query")

/lustre/scratch124/cellgen/haniffa/projects/adult_skin_v1/nobackup_data/adata_core_plus_RELAPSE_refquery2.h5ad.svg.query


In [56]:
"""
this adata can now be used as input for the ref-query tutorial for
nichecompass
https://nichecompass.readthedocs.io/en/latest/tutorials/notebooks/mouse_cns_spatial_reference_mapping.html

we apply this in a python job in ../job_scripts/nichecompass_ref_mapping2.py

"""

'\nthis adata can now be used as input for the ref-query tutorial for\nnichecompass\nhttps://nichecompass.readthedocs.io/en/latest/tutorials/notebooks/mouse_cns_spatial_reference_mapping.html\n\nwe apply this in a python job in ../job_scripts/nichecompass_ref_mapping2.py\n\n'

In [57]:
2

2

In [58]:
.

SyntaxError: invalid syntax (1933637684.py, line 1)

# Results

In [ ]:
PATH_RES = '/lustre/scratch124/cellgen/haniffa/projects/developmental_fibroblasts/nobackup_output/nichecompasss/nichecompass/artifacts/spatial_reference_mapping/20251201_091037XeniumTUTORIAL__REFQ/model/' 


In [ ]:
import os
os.listdir(PATH_RES + "reference")

In [ ]:
PATH_RES + "reference"

In [ ]:
import scanpy as sc
adata=sc.read_h5ad ( PATH_RES + "reference_query/adata.h5ad")
adata.obs["batch_nc"].value_counts()

In [ ]:
# import pickle

# outpath = "/nfs/team298/ls34/skin_niche19.pkl"

# with open(outpath, "rb") as f:
#     md_loaded = pickle.load(f)

# adata.obs["niche19"] = adata.obs.index.map(md_loaded)
# adata.obs["niche19"]=adata.obs["niche19"].fillna("QUERY_DATA")

In [ ]:
sc.settings.set_figure_params(dpi_save=300, facecolor="white", frameon=False, figsize=(20,20))

sc.pl.umap(
    adata,
    color=[
        "batch_nc",
    ],
    #legend_loc="on data",
    s=10,
    legend_fontsize=54,
    legend_fontoutline=2,
    show=False,
    #palette=colorsmap,
    edgecolor='black',
    linewidth=0.01,
    title='',
    palette=['lightblue', 'orange']
#   save=f"{x}_PAGA.pdf"
)

In [ ]:
colors_new2 = {
        'Epidermis_basal': "#4a538f",# "#3489f1", #"#000080", #"#f0f8ff", #"#007cfe",#'#4f59a1',  # dark blue
    "Epidermis basal":"#3489f1",# "#000080", #"#f0f8ff", #"#000080",
    
                'Epidermis_mid': '#F0F8FF', # #e3e7eb - GREY
  #  "Epidermis_late": "#4682B4", #"#A52A2A",          # pink
            'Epidermis_late':"#2a2f5c",#"#2a2f5c" ,# "#2e2e2e",#'#b1dae6',  # light blue

   'Epidermis basal_cycling': "#007cfe",  #  e2d8dd   "#505aa1",#"#c590a5" , # '#e0c5d2',
'Epidermis_basal_cycling': "#007cfe", 
'EpidermisInflamm_mid':"#f6e3ec",
        'EpidermisInflamm_late': "#800020",
    
    
    # 'Epidermis_Inflammatory': '#f2e7ed',
        "Epidermis_APChi": "#ff1493",
    'Papillary_dermis':"#FFFF00", # "#f6d17a", # yellow
    
  #  'Perivascular':  "#845DAE", #  '#fedbe7' PINKSIH,  #'#f8daeb',  # darkish red
    'Small blood vessel': "#d73435", #'#fe78bb',  # bright red
    'Sweat_gland': '#008080',  # dark grey
        'Sweat_gland_channel': "#adf7e8", #'#008080',  # original - 40E0D0
    'Muscle': "#f371af",  # very dark red
    'Adipocyte+vessels': "#B8860B", #'#FF4500',  # very dark orange
     'Hypodermis': "#FFFFF0", #"#D2B48C",#' # yello  # white = '#ffffff',
    'Larger blood vessel': "#660000",#"#8B0000", #'#FF6347',  # coral red
    'Perineural': '#0A0A0A',  # almost black
   # 'Plasma+pDC': '#808080',  # bright turquoise
    #'Sweat gland channel': '#000000',  # black
    #'KC_immunecell': '#FF1493',  # dark pink
    
   # 'Epidermis_late': '#b1dae6',
    "Nonspecific":"#D3D3D3",
     'Reticular_dermis': '#d0e1f2',

 'Large_BV': '#660000',
 'Small_BV': "#c43f40",#'#D96B6B',
     'Tzone-like': "#845DAE",# '#F4D1A1',

 'Sebaceous_gland': "#FFD9A3",      #  "#FFB347",#"#94cb72",#'#e28743',
"Sebaceous_immune": "#ff6145", 
 #'Plasma_cell_niche': "#1F51FF", #"#FFB300",#"#40E0D0",#'#ff5e00',
   # 'Plasma_cell_rich':  "#8080B2", #"
"Plasma_cell_rich": "#04D9FF",
    'HF_Perineural': '#00FF00',
    "OuterHF": "#006600",# 
 'Sebaceous duct': '#e28743',


"Lymphoid Tzone-like": "#825bac",
    



     #007FFF
    "Reticular_dermis_LE_rich" : "#6A8ED8",
        "Reticular_dermis_LErich" : "#6A8ED8",
    "Small_BV_Trich": "#FFB6C1",          # neon orange
    "Tzone-like_Theavy": "#B19CD9",       # light purple
    #"Reticular dermis_F2/F3hi": "#35476C", #"#FF5F1F",# light blue
    "HF_outer": "#90c670",                # very dark green
    "VenuleMuscle": "#B34A7E",

    "Sebaceous_duct": "#FFA07A",          # light orange (salmony)
    "VenuleEndothelium": "#8B0000",       # dark red
    "HF_inner": "#90EE90",                 # light green
    "HF_TNN+COCH+hi": "#39FF14",          # neon green
    # "Peri_sweat_gland": "#d9f9f7",
    'HF_innermost':  "#D95D54",
  #  'Sebaceous_inner_immune':"#F6A85C",
    'Nonspecific/folded': "#F0F0F0",
 'nan': '#FAFAFA',
    "QUERY_DATA": '#F0F0F0'
    }


In [ ]:
"""
how do niches compare to what we had before
"""
sc.pl.umap(
    adata,
    color=[
        "niche19",
    ],
    legend_loc="on data",
    s=5,
    legend_fontsize=14,
    legend_fontoutline=2,
    show=False,
    #palette=colorsmap,
    edgecolor='black',
    linewidth=0.01,
    title='',
    palette=colors_new2,
#   save=f"{x}_PAGA.pdf"
)

In [ ]:
"""
how do niches compare to what we had before
"""
sc.pl.umap(
    adata,
    color=[
        "niche19",
    ],
   # legend_loc="on data",
    s=5,
    legend_fontsize=14,
    legend_fontoutline=2,
    show=False,
    #palette=colorsmap,
    edgecolor='black',
    linewidth=0.01,
    title='',
    palette=colors_new2,
#   save=f"{x}_PAGA.pdf"
)

In [ ]:
# adata.write( PATH_RES + "reference_query/adata.h5ad")
